# SCRUM-10: Finalize Dataset Documentation

**Task:** Compile the formal dataset documentation detailing all collection methods and source constraints.

**Assignee:** Mounik Sai | **Sprint:** SCRUM Sprint 0 / Review 1

---

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

DATA_DIR = os.path.join('..', 'data')
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')
YT_DIR = os.path.join(DATA_DIR, 'youtube')
DOCS_DIR = os.path.join('..', 'docs')
print('Environment initialized successfully.')

Environment initialized successfully.


## 1. Product Registry & Target Scope Auditing

Auditing the 60 smartphone registry across brands, categories, and lifecycle classifications.

In [2]:
products = pd.read_csv(os.path.join(DATA_DIR, 'products_v3.csv'))
print(f'Total Products in Active Registry: {len(products)}')
print(f'Unique Brands: {products["brand"].nunique()}')
print(f'Categories: {products["category"].unique().tolist()}')
print('\n--- Product Lifecycle Breakdown ---')
print(products['product_type'].value_counts())

Total Products in Active Registry: 60
Unique Brands: 13
Categories: ['Smartphones']

--- Product Lifecycle Breakdown ---
product_type
RISING       24
STABLE       24
DECLINING    12
Name: count, dtype: int64


In [3]:
brand_summary = products.groupby(['brand', 'product_type']).size().unstack(fill_value=0)
brand_summary['Total'] = brand_summary.sum(axis=1)
brand_summary = brand_summary.sort_values(by='Total', ascending=False)
print('--- Brand by Product Type Distribution ---')
print(brand_summary)

--- Brand by Product Type Distribution ---
product_type  DECLINING  RISING  STABLE  Total
brand                                         
Samsung               3       4       6     13
Apple                 4       3       2      9
Google                2       3       1      6
OnePlus               1       1       3      5
Redmi                 1       3       1      5
Vivo                  1       2       2      5
Oppo                  0       2       2      4
Nothing               0       1       2      3
Poco                  0       2       1      3
Realme                0       0       2      2
Motorola              0       1       1      2
iQOO                  0       1       1      2
Infinix               0       1       0      1


## 2. Multi-Source Collection Architecture & Audit Logs

Examining `source_log.csv` and `collection_log.csv` to document all ingestion mechanisms and parameters.

In [4]:
source_log = pd.read_csv(os.path.join(DATA_DIR, 'source_log.csv'))
print('--- Data Sources Configured ---')
for idx, row in source_log.iterrows():
    print(f"{idx+1}. {row['source']} [{row['added_date']}]: {row['granularity']}")
    print(f"   Method: {row['access_method'][:90]}...")

--- Data Sources Configured ---
1. YouTube Data API v3 [2026-10-03]: daily snapshot per product (counts are cumulative at collection time)
   Method: Official REST API with an API key (src/collect_youtube.py); search.list then videos.list t...
2. Google Trends [2026-10-03]: weekly for a 12-month window (daily for windows of 90 days or less)
   Method: Manual CSV export from trends.google.com (Interest over time), saved untouched to raw/tren...
3. Google Trends [2026-10-03]: daily; window changed on 2026-10-03 from 12 months weekly to a 269-day custom range (about 9 months), identical for every product; supersedes the earlier Google Trends row
   Method: Manual CSV export from trends.google.com (Interest over time) with a custom time range, sa...
4. YouTube Data API v3 [2026-10-03]: one snapshot per product per run day; recent pass counts uploads in a rolling 7-day window
   Method: Official REST API with an API key (src/collect_youtube.py). Main pass: search.list (order=...
5. Google T

In [5]:
collection_log = pd.read_csv(os.path.join(DATA_DIR, 'collection_log.csv'))
print(f'Total Collection Operations Logged: {len(collection_log):,}')
print('\nLog Records by Source:')
print(collection_log['source'].value_counts())
print('\nLog Records by Status:')
print(collection_log['status'].value_counts())

Total Collection Operations Logged: 412

Log Records by Source:
source
google_trends_pytrends    194
youtube                    78
youtube_recent             70
google_trends              70
Name: count, dtype: int64

Log Records by Status:
status
success              271
skipped_cached       136
error                  3
smoke_test_ok          1
comments_disabled      1
Name: count, dtype: int64


## 3. Google Trends Data Audit & Source Constraints

Validating daily search interest index, date range boundaries, provisional status, and zero-share distributions.

In [6]:
trends = pd.read_csv(os.path.join(PROCESSED_DIR, 'trends_clean.csv'))
print(f'Trends Total Rows: {len(trends):,}')
print(f'Distinct Products: {trends["product_id"].nunique()}')
print(f'Date Span: {trends["date"].min()} to {trends["date"].max()} ({trends["date"].nunique()} days)')
zero_pct = (trends['is_zero'] == True).mean() * 100
print(f'Overall Zero-Interest Day Percentage: {zero_pct:.2f}%')

Trends Total Rows: 16,140
Distinct Products: 60
Date Span: 2026-01-08 to 2026-10-03 (269 days)
Overall Zero-Interest Day Percentage: 22.48%


## 4. YouTube Data API v3 Audit & Constraint Analysis

Auditing video volume, snapshot dates, comment volumes, and missing value occurrences.

In [7]:
videos = pd.read_csv(os.path.join(PROCESSED_DIR, 'youtube_videos_clean.csv'))
comments = pd.read_csv(os.path.join(PROCESSED_DIR, 'youtube_comments_clean.csv'))
daily = pd.read_csv(os.path.join(PROCESSED_DIR, 'youtube_product_daily_clean.csv'))

print(f'Cleaned Videos: {len(videos):,} across {videos["product_id"].nunique()} products')
print(f'Cleaned Comments: {len(comments):,} across {comments["product_id"].nunique()} products')
print(f'Snapshot Daily Records: {len(daily)} across snapshot dates: {daily["snapshot_date"].unique().tolist()}')

Cleaned Videos: 3,500 across 60 products
Cleaned Comments: 32,215 across 60 products
Snapshot Daily Records: 70 across snapshot dates: ['2026-10-03', '2026-10-04']


In [8]:
print('--- Video Level Missing Values ---')
print(videos[['view_count', 'like_count', 'comment_count']].isna().sum())
print('\n--- Comment Level Flags ---')
print(f'Emoji-only comments: {comments["is_emoji_only"].sum()} ({comments["is_emoji_only"].mean():.2%})')
print(f'Likely non-English (non-Latin script): {comments["likely_non_english"].sum()} ({comments["likely_non_english"].mean():.2%})')

--- Video Level Missing Values ---
view_count        0
like_count       85
comment_count     4
dtype: int64

--- Comment Level Flags ---
Emoji-only comments: 395 (1.23%)
Likely non-English (non-Latin script): 934 (2.90%)


## 5. Precision & Sibling Model Contamination (`on_target_share`)

Evaluating title matching precision to assess search query bleed from sibling device models.

In [9]:
report = pd.read_csv(os.path.join(PROCESSED_DIR, 'full_report.csv'))
low_target = report[report['low_on_target_share'] == True][['product_id', 'product_name', 'on_target_share', 'issues']]
print(f'Products with On-Target Share < 60%: {len(low_target)}')
print(low_target.to_string(index=False))

Products with On-Target Share < 60%: 4
product_id        product_name  on_target_share                                                 issues
      P009 Google Pixel 11 Pro             0.50                              low on-target share (50%)
      P012   Redmi Note 17 Pro             0.56 low signal (zero share 54%), low on-target share (56%)
      P015             Poco X8             0.48                              low on-target share (48%)
      P028  Samsung Galaxy S26             0.56                              low on-target share (56%)


## 6. Multi-Source Alignment & Usability Audit

Reviewing `aligned_daily.csv` and `full_report.csv` for end-to-end dataset usability.

In [10]:
aligned = pd.read_csv(os.path.join(PROCESSED_DIR, 'aligned_daily.csv'))
print(f'Aligned Table Dimensions: {aligned.shape[0]:,} rows x {aligned.shape[1]} columns')
print(f'Trends Observed: {aligned["trends_observed"].sum():,} / {len(aligned):,} ({aligned["trends_observed"].mean():.1%})')
print(f'YouTube Observed: {aligned["youtube_observed"].sum():,} / {len(aligned):,}')
print(f'Usable Products Flagged in Full Report: {report["usable"].sum()} / {len(report)}')
print('\n=== SCRUM-10 DATASET DOCUMENTATION AUDIT COMPLETE ===')

Aligned Table Dimensions: 16,140 rows x 28 columns
Trends Observed: 16,140 / 16,140 (100.0%)
YouTube Observed: 10 / 16,140
Usable Products Flagged in Full Report: 42 / 60

=== SCRUM-10 DATASET DOCUMENTATION AUDIT COMPLETE ===
